In [1]:
import pandas as pd
import json
import os
from tqdm import tqdm
from glob import glob

In [2]:
class ner_eval():
    def __init__(self, true_list, pred_list, eval_dict, inv=False, sentence_eval_dict=None):
        """
        Initialize ner_eval class for both entity-level and sentence-level evaluation.
        """
        self.true_list = true_list
        self.pred_list = pred_list
        self.eval_dict = eval_dict  # For strict/relax entity-level evaluation
        self.inv = inv  # Whether to reverse true and pred lists
        self.true_length = len(true_list)
        self.pred_length = len(pred_list)
        self.check = self.true_length == self.pred_length  # Validate input lengths
        
        # Define categories once as a class attribute
        self.categories = [
            'Adherence', 'Concern', 'Education', 'Employment', 'Financial', 
            'Healthcare', 'Insurance', 'Literacy', 'Living', 'MentalHealth', 
            'Recommendation', 'Smoke', 'Social', 'SubstanceUse', 'Transportation', 'Trauma'
        ]
        
        self.set_info()  # Process BIO tags to get entity spans
        
        # Initialize sentence-level evaluation dictionary
        self.sentence_eval_dict = sentence_eval_dict if sentence_eval_dict is not None else {}

    def get_info(self):
        """
        Extract entity spans (start idx, end idx, entity type) from BIO tags.
        """
        s_idx_list = []
        e_idx_list = []
        ent_list = []
        true_list = self.pred_list if self.inv else self.true_list
        
        if self.check:
            cnt = -1
            for i in range(len(true_list)):
                if true_list[i] != 'O':
                    label = true_list[i].split('-')
                    if label[0] == 'B':
                        cnt += 1
                        s_idx_list.append(i)
                        e_idx_list.append(i+1)
                        ent_list.append(label[1])
                    elif label[0] == 'I':
                        try:
                            prev = true_list[i-1]
                            if prev == 'O' or prev.split('-')[1] != label[1]:
                                cnt += 1
                                s_idx_list.append(i)
                                e_idx_list.append(i+1)
                                ent_list.append(label[1])
                            else:
                                e_idx_list[cnt] = i+1
                        except IndexError:
                            cnt += 1
                            s_idx_list.append(i)
                            e_idx_list.append(i+1)
                            ent_list.append(label[1])
        return s_idx_list, e_idx_list, ent_list

    def set_info(self):
        if self.check:
            self.s_idx_list, self.e_idx_list, self.ent_list = self.get_info()

    def strict(self, true, pred, s_idx, e_idx, entity):
        if true[s_idx] != f'B-{entity}' or pred[s_idx] != f'B-{entity}':
            return False
        for idx in range(s_idx, e_idx):
            if true[idx] != pred[idx]:
                return False
        if e_idx < len(true) and (true[e_idx] == f'I-{entity}' or pred[e_idx] == f'I-{entity}'):
            return False
        return True

    def relax(self, true, pred, s_idx, e_idx, entity):
        for idx in range(s_idx, e_idx):
            try:
                true_ent = true[idx].split('-')[1]
                pred_ent = pred[idx].split('-')[1]
                if true_ent == pred_ent == entity:
                    return True
            except IndexError:
                continue
        return False        

    def get_strict(self, s_idx, e_idx, ent):
        return self.strict(self.pred_list, self.true_list, s_idx, e_idx, ent) if self.inv else self.strict(self.true_list, self.pred_list, s_idx, e_idx, ent)

    def get_relax(self, s_idx, e_idx, ent):
        return self.relax(self.pred_list, self.true_list, s_idx, e_idx, ent) if self.inv else self.relax(self.true_list, self.pred_list, s_idx, e_idx, ent)

    def sentence_eval(self):
        result = {}
        for i in range(len(self.s_idx_list)):
            s_idx = self.s_idx_list[i]
            e_idx = self.e_idx_list[i]
            ent = self.ent_list[i]
            strict = self.get_strict(s_idx, e_idx, ent)
            relax = self.get_relax(s_idx, e_idx, ent)
            
            if ent in result:
                result[ent]['strict'] += 1 if strict else 0
                result[ent]['relax'] += 1 if relax else 0
                result[ent]['total'] += 1
            else:
                result[ent] = {'strict': int(strict), 'relax': int(relax), 'total': 1}
        return result

    def update_dict(self):
        if self.check:
            result = self.sentence_eval()
            for k, v in result.items():
                if k in self.eval_dict:
                    self.eval_dict[k]['strict'] += v['strict']
                    self.eval_dict[k]['relax'] += v['relax']
                    self.eval_dict[k]['total'] += v['total']
                else:
                    self.eval_dict[k] = v.copy()  # Added .copy() to prevent reference bugs
            return self.eval_dict
        else:
            return self.eval_dict

    def sum_and_fulfilldict(self):
        missing_category = list(set(self.categories) - set(self.eval_dict.keys()))
        for i in missing_category:
            self.eval_dict[i] = {'strict': 0, 'relax': 0, 'total': 0}
            
        strict, relax, total = 0, 0, 0
        for v in self.eval_dict.values():
            if isinstance(v, dict) and 'strict' in v: # Avoid counting the 'overall' key if run twice
                strict += v['strict']
                relax += v['relax']
                total += v['total']
                
        self.eval_dict['overall'] = {'strict': strict, 'relax': relax, 'total': total}
        return dict(sorted(self.eval_dict.items()))

    ### ------------------- Sentence-Level Evaluation ------------------- ###
    def update_sentence_level_dict(self):
        if not self.check:
            return self.sentence_eval_dict
        
        true_entities = set([tag.split('-')[-1] for tag in self.true_list if tag.startswith(('B-', 'I-'))])
        pred_entities = set([tag.split('-')[-1] for tag in self.pred_list if tag.startswith(('B-', 'I-'))])

        for cat in self.categories:
            if cat not in self.sentence_eval_dict:
                self.sentence_eval_dict[cat] = {'TP': 0, 'FP': 0, 'FN': 0}

            if cat in true_entities and cat in pred_entities:
                self.sentence_eval_dict[cat]['TP'] += 1
            if cat not in true_entities and cat in pred_entities:
                self.sentence_eval_dict[cat]['FP'] += 1
            if cat in true_entities and cat not in pred_entities:
                self.sentence_eval_dict[cat]['FN'] += 1

        return self.sentence_eval_dict

    def compute_sentence_level_metrics(self):
        sentence_metrics = {}
        for cat in self.categories:
            if cat not in self.sentence_eval_dict:
                TP = FP = FN = 0
            else:
                TP = self.sentence_eval_dict[cat]['TP']
                FP = self.sentence_eval_dict[cat]['FP']
                FN = self.sentence_eval_dict[cat]['FN']

            precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
            recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
            f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

            sentence_metrics[cat] = {
                'sentence-level': {'precision': round(precision,3), 'recall': round(recall,3), 'f-1': round(f1,3)}
            }
        return sentence_metrics
    
    def compute_entity_level_metrics_separated(self, true_eval_dict, pred_eval_dict):
        metrics = {}
        for cat in self.categories:
            # Strict
            tp_strict = true_eval_dict[cat]['strict']
            total_true = true_eval_dict[cat]['total']
            total_pred = pred_eval_dict[cat]['total']

            strict_precision = tp_strict / total_pred if total_pred > 0 else 0.0
            strict_recall = tp_strict / total_true if total_true > 0 else 0.0
            strict_f1 = (2 * strict_precision * strict_recall) / (strict_precision + strict_recall) if (strict_precision + strict_recall) > 0 else 0.0

            # Relax (FIXED: Using separated tp_relax values)
            tp_relax_recall = true_eval_dict[cat]['relax']
            tp_relax_precision = pred_eval_dict[cat]['relax']
            
            relax_precision = tp_relax_precision / total_pred if total_pred > 0 else 0.0
            relax_recall = tp_relax_recall / total_true if total_true > 0 else 0.0
            relax_f1 = (2 * relax_precision * relax_recall) / (relax_precision + relax_recall) if (relax_precision + relax_recall) > 0 else 0.0

            metrics[cat] = {
                'strict': {'precision': round(strict_precision,3), 'recall': round(strict_recall,3), 'f-1': round(strict_f1,3)},
                'relax': {'precision': round(relax_precision,3), 'recall': round(relax_recall,3), 'f-1': round(relax_f1,3)}
            }
        return metrics

    def compute_overall_metrics_separated(self, true_eval_dict, pred_eval_dict, sentence_eval_dict):
        # ---- Entity-level Overall ----
        strict_tp_sum = sum([true_eval_dict[cat]['strict'] for cat in self.categories])
        strict_total_true = sum([true_eval_dict[cat]['total'] for cat in self.categories])
        strict_total_pred = sum([pred_eval_dict[cat]['total'] for cat in self.categories])
        
        # FIXED: Relax counts separately for true and pred
        relax_tp_sum_true = sum([true_eval_dict[cat]['relax'] for cat in self.categories])
        relax_tp_sum_pred = sum([pred_eval_dict[cat]['relax'] for cat in self.categories])
        
        relax_total_true = strict_total_true # Total counts remain the same
        relax_total_pred = strict_total_pred

        # Strict Overall Math
        strict_precision = strict_tp_sum / strict_total_pred if strict_total_pred > 0 else 0.0
        strict_recall = strict_tp_sum / strict_total_true if strict_total_true > 0 else 0.0
        strict_f1 = (2 * strict_precision * strict_recall) / (strict_precision + strict_recall) if (strict_precision + strict_recall) > 0 else 0.0

        # Relax Overall Math
        relax_precision = relax_tp_sum_pred / relax_total_pred if relax_total_pred > 0 else 0.0
        relax_recall = relax_tp_sum_true / relax_total_true if relax_total_true > 0 else 0.0
        relax_f1 = (2 * relax_precision * relax_recall) / (relax_precision + relax_recall) if (relax_precision + relax_recall) > 0 else 0.0

        # ---- Sentence-level Overall ----
        sentence_TP_sum = sum([sentence_eval_dict[cat]['TP'] for cat in self.categories if cat in sentence_eval_dict])
        sentence_FP_sum = sum([sentence_eval_dict[cat]['FP'] for cat in self.categories if cat in sentence_eval_dict])
        sentence_FN_sum = sum([sentence_eval_dict[cat]['FN'] for cat in self.categories if cat in sentence_eval_dict])

        sentence_precision = sentence_TP_sum / (sentence_TP_sum + sentence_FP_sum) if (sentence_TP_sum + sentence_FP_sum) > 0 else 0.0
        sentence_recall = sentence_TP_sum / (sentence_TP_sum + sentence_FN_sum) if (sentence_TP_sum + sentence_FN_sum) > 0 else 0.0
        sentence_f1 = (2 * sentence_precision * sentence_recall) / (sentence_precision + sentence_recall) if (sentence_precision + sentence_recall) > 0 else 0.0

        return {
            'strict': {'precision': round(strict_precision,3), 'recall': round(strict_recall,3), 'f-1': round(strict_f1,3)},
            'relax': {'precision': round(relax_precision,3), 'recall': round(relax_recall,3), 'f-1': round(relax_f1,3)},
            'sentence-level': {'precision': round(sentence_precision,3), 'recall': round(sentence_recall,3), 'f-1': round(sentence_f1,3)}
        }

In [3]:
def fix_bio_tags(bio_lists):
    fixed = []
    for tag_list in bio_lists:
        tag_list = ["O" if x == "" else x for x in tag_list]
        new_tags = []
        for i, tag in enumerate(tag_list):
            # If tag is 'O', nothing to change.
            if tag == 'O':
                new_tags.append(tag)
            elif tag == "":
                new_tags.append("O")
            else:

                prefix, entity = tag.split('-', 1)

                # If first token, or previous token is 'O', or previous token is a different entity,
                # then this token should be a beginning (B-) tag.

                if i == 0 or tag_list[i-1] == 'O' or (tag_list[i-1] != 'O' and tag_list[i-1].split('-', 1)[1] != entity):
                    new_tags.append('B-' + entity)
                else:
                    # Otherwise, continue the entity span as an inside (I-) tag.
                    new_tags.append('I-' + entity)
        fixed.append(new_tags)
    return fixed

def refine_bio_tags(bio_lists):
    refined = []
    for tags in bio_lists:
        new_tags = tags[:]  # work on a copy
        # Iterate from the second token to the second-to-last token.
        for i in range(1, len(new_tags) - 1):
            # Look for an "O" token.
            if new_tags[i] == 'O':
                prev_tag = new_tags[i - 1]
                next_tag = new_tags[i + 1]
                # Check if the previous token is part of an entity and the next token starts an entity.
                if prev_tag != 'O' and next_tag.startswith('B-'):
                    prev_entity = prev_tag.split('-', 1)[1]
                    next_entity = next_tag.split('-', 1)[1]
                    # If both tokens refer to the same entity, fill in the gap and adjust the following token.
                    if prev_entity == next_entity:
                        new_tags[i] = 'I-' + prev_entity
                        new_tags[i + 1] = 'I-' + next_entity
        refined.append(new_tags)
    return refined


In [4]:
model_results = {} # Assuming this exists outside the loop based on your original code

for model_name in ['BERT', 'BioBERT', 'RoBERTa']:
# for model_name in ['RoBERTa']:
    lr = "5e-5"
    model_folds_metrics = {}
    for fold in range(1, 6):
    #     if model_name == "BERT":
        #     d = f'../output/{model_name}_fold_{fold}_lr_3e-5/predictions/'
        # else:
        d = f'../output/{model_name}_fold_{fold}_lr_{lr}/predictions/'
        y_true = []
        with open(f'../data/splitted_data/fold_{fold}/test.json', 'r') as textfile:
            for i in json.load(textfile):
                y_true.append(i['ner_tags'])

        predict_file = d + 'predictions.txt'
        y_pred = []
        with open(predict_file) as txtfile:
            for i in txtfile.readlines():
                y_pred.append(i.split())
        y_pred = refine_bio_tags(fix_bio_tags(y_pred))

        # ----- Initialize two entity dicts -----
        eval_dict_true = {}
        eval_dict_pred = {}

        # ----- Initialize sentence-level dict -----
        sentence_eval_dict = {}

        # --------- First pass: y_true reference ---------
        for i in range(len(y_true)):
            eval_dict_true = ner_eval(y_true[i], y_pred[i], eval_dict_true).update_dict()
            sentence_eval_dict = ner_eval(y_true[i], y_pred[i], eval_dict_true, sentence_eval_dict=sentence_eval_dict).update_sentence_level_dict()
        eval_dict_true = ner_eval(y_true[i], y_pred[i], eval_dict_true).sum_and_fulfilldict()

        # --------- Second pass: y_pred reference (inv=True) ---------
        for i in range(len(y_true)):
            eval_dict_pred = ner_eval(y_true[i], y_pred[i], eval_dict_pred, inv=True).update_dict()
        eval_dict_pred = ner_eval(y_true[i], y_pred[i], eval_dict_pred, inv=True).sum_and_fulfilldict()

        # --------- Compute metrics ---------
        ner_instance = ner_eval(y_true[0], y_pred[0], eval_dict_true, sentence_eval_dict=sentence_eval_dict)

        # Compute correct entity-level metrics (passing both dicts!)
        entity_metrics = ner_instance.compute_entity_level_metrics_separated(eval_dict_true, eval_dict_pred)

        # Compute sentence-level metrics
        sentence_metrics = ner_instance.compute_sentence_level_metrics()

        # --------- Combine strict, relax, sentence-level ---------
        final_metrics = {}
        categories = entity_metrics.keys()
        for cat in categories:
            final_metrics[cat] = entity_metrics[cat]
            final_metrics[cat]['sentence-level'] = sentence_metrics[cat]['sentence-level']

        # --------- Add overall (optional) ---------
        overall_metrics = ner_instance.compute_overall_metrics_separated(eval_dict_true, eval_dict_pred, sentence_eval_dict)
        final_metrics['overall'] = overall_metrics

        # Save fold result
        model_folds_metrics[f"fold_{fold}"] = final_metrics

        # Save model result
        model_results[model_name] = model_folds_metrics



In [5]:
for model_name in ["BiLSTM-CRF", "Ctran"]:
    model_folds_metrics = {}
    y_true = []
    
    for fold in range(1, 6):
        d = f'../output/{model_name}_fold_{fold}_lr_5e-5/predictions/'
        y_true = []
        with open(f'../data/splitted_data/fold_{fold}/test.json', 'r') as textfile:
            for i in json.load(textfile):
                y_true.append(i['ner_tags'])

        y_pred = []
        if model_name == "BiLSTM-CRF":
            predict_file = f"../output/fold_{fold}/test.txt_predict.txt"
            with open(predict_file) as txtfile:
                for i in txtfile.readlines():
                    # FIXED: Skip empty lines to prevent misalignment
                    if i.strip(): 
                        y_pred.append([x.split("/")[1] for x in i.split()])

        elif model_name == "Ctran":
            predict_file = f"../output/fold_{fold}/test_predictions.csv"
            data = pd.read_csv(predict_file)
            y_pred = [x.split() for x in data["Tag Prediction"].tolist()]

        y_pred = refine_bio_tags(fix_bio_tags(y_pred))

        # ----- Initialize dicts cleanly -----
        eval_dict_true = {}
        eval_dict_pred = {}
        sentence_eval_dict = {}

        # --------- Processing Loop ---------
        # Iterate through the lists simultaneously
        for i in range(len(y_true)):
            # True Reference (Normal)
            ner_true = ner_eval(y_true[i], y_pred[i], eval_dict_true, sentence_eval_dict=sentence_eval_dict)
            eval_dict_true = ner_true.update_dict()
            sentence_eval_dict = ner_true.update_sentence_level_dict()

            # Pred Reference (Inverted)
            ner_pred = ner_eval(y_true[i], y_pred[i], eval_dict_pred, inv=True)
            eval_dict_pred = ner_pred.update_dict()

        # Fulfill the final dictionaries once at the end, not inside the loop
        # Using the last created instances to call the fulfillment method
        eval_dict_true = ner_true.sum_and_fulfilldict()
        eval_dict_pred = ner_pred.sum_and_fulfilldict()

        # --------- Compute metrics ---------
        # Compute correct entity-level metrics
        entity_metrics = ner_true.compute_entity_level_metrics_separated(eval_dict_true, eval_dict_pred)
        sentence_metrics = ner_true.compute_sentence_level_metrics()

        # Combine strict, relax, sentence-level
        final_metrics = {}
        for cat in ner_true.categories:
            final_metrics[cat] = entity_metrics[cat]
            final_metrics[cat]['sentence-level'] = sentence_metrics[cat]['sentence-level']

        # Add overall metrics
        overall_metrics = ner_true.compute_overall_metrics_separated(eval_dict_true, eval_dict_pred, sentence_eval_dict)
        final_metrics['overall'] = overall_metrics

        # Save fold result
        model_folds_metrics[f"fold_{fold}"] = final_metrics

        # Save model result
        model_results[model_name] = model_folds_metrics

In [6]:
def average_metrics(fold_metrics_dict):
    avg_metrics = {}
    num_folds = len(fold_metrics_dict)
    
    categories = fold_metrics_dict["fold_1"].keys()

    for cat in categories:
        avg_metrics[cat] = {'strict': {'precision': 0, 'recall': 0, 'f-1': 0},
                            'relax': {'precision': 0, 'recall': 0, 'f-1': 0},
                            'sentence-level': {'precision': 0, 'recall': 0, 'f-1': 0}}
        
        # Sum across folds
        for fold_num, fold_metric in fold_metrics_dict.items():
            for metric_type in ['strict', 'relax', 'sentence-level']:
                # print(fold_metrics_list)
                avg_metrics[cat][metric_type]['precision'] += fold_metric[cat][metric_type]['precision']
                avg_metrics[cat][metric_type]['recall'] += fold_metric[cat][metric_type]['recall']
                avg_metrics[cat][metric_type]['f-1'] += fold_metric[cat][metric_type]['f-1']
        
        # Average
        for metric_type in ['strict', 'relax', 'sentence-level']:
            avg_metrics[cat][metric_type]['precision'] = round(avg_metrics[cat][metric_type]['precision'] / num_folds, 3)
            avg_metrics[cat][metric_type]['recall'] = round(avg_metrics[cat][metric_type]['recall'] / num_folds, 3)
            avg_metrics[cat][metric_type]['f-1'] = round(avg_metrics[cat][metric_type]['f-1'] / num_folds, 3)
    
    return avg_metrics


In [7]:
final_avg_results = {}

for model_name in model_results.keys():
    avg_metrics = average_metrics(model_results[model_name])
    final_avg_results[model_name] = avg_metrics

In [8]:
for k,v in model_results.items():
    v['average'] = final_avg_results[k]

In [9]:
import pandas as pd

def convert_overall_to_dataframe(final_avg_results):
    rows = []
    for model_name, metrics in final_avg_results.items():
        row = {'Model': model_name}
        for eval_type in ['strict', 'relax', 'sentence-level']:
            row[f'{eval_type}_precision'] = metrics['overall'][eval_type]['precision']
            row[f'{eval_type}_recall'] = metrics['overall'][eval_type]['recall']
            row[f'{eval_type}_f1'] = metrics['overall'][eval_type]['f-1']
        rows.append(row)
    df = pd.DataFrame(rows)
    return df
df = convert_overall_to_dataframe(final_avg_results)

In [16]:
df.sort_values(by='sentence-level_f1',ascending=False).reset_index(drop=True)

,Model,strict_precision,strict_recall,strict_f1,relax_precision,relax_recall,relax_f1,sentence-level_precision,sentence-level_recall,sentence-level_f1
0,RoBERTa,0.486,0.486,0.485,0.673,0.682,0.677,0.746,0.732,0.738
1,BERT,0.480,0.466,0.471,0.669,0.658,0.661,0.747,0.705,0.724
2,BioBERT,0.473,0.466,0.469,0.654,0.649,0.651,0.741,0.700,0.719
3,Ctran,0.436,0.440,0.438,0.615,0.632,0.623,0.691,0.681,0.685
4,BiLSTM-CRF,0.107,0.018,0.031,0.396,0.065,0.110,0.460,0.074,0.127


In [10]:
from datetime import datetime
# Get today's date and format it as mmddyy
formatted_date = datetime.now().strftime("%m%d%y")

In [15]:
from pathlib import Path
dir_path = Path(f"../results/{formatted_date}")
dir_path.mkdir(parents=True, exist_ok=True)

In [16]:
with open(f"../results/{formatted_date}/model_results.json","w") as f:
    json.dump(model_results,f)

In [17]:
# Save to Excel
df.to_excel(f'../results/{formatted_date}/model_overall_summary.xlsx', index=False)
df.to_csv(f'../results/{formatted_date}/model_overall_summary.csv')

In [18]:
df

,Model,strict_precision,strict_recall,strict_f1,relax_precision,relax_recall,relax_f1,sentence-level_precision,sentence-level_recall,sentence-level_f1
0,BERT,0.480,0.466,0.471,0.669,0.658,0.661,0.747,0.705,0.724
1,BioBERT,0.473,0.466,0.469,0.654,0.649,0.651,0.741,0.700,0.719
2,RoBERTa,0.486,0.486,0.485,0.673,0.682,0.677,0.746,0.732,0.738
3,BiLSTM-CRF,0.107,0.018,0.031,0.396,0.065,0.110,0.460,0.074,0.127
4,Ctran,0.436,0.440,0.438,0.615,0.632,0.623,0.691,0.681,0.685


In [19]:
model_results = json.load(open(f"../results/{formatted_date}/model_results.json","r"))

In [20]:
average_roberta = []
for tag,info in model_results["RoBERTa"]['average'].items():
    tmp = []
    tmp.append(tag)
    for t, value in info.items():
        tmp += [value['precision'], value['recall'],value['f-1']]
    average_roberta.append(tmp)

In [21]:
roberta_df = pd.DataFrame(average_roberta, columns = ["Tag","SP","SR","SF1","RP","RR","RF1","SP","SR","SF1"])

In [22]:
roberta_df.to_csv(f"../results/{formatted_date}/best_model_tag_performance.csv")

In [5]:
roberta_df = pd.read_csv("../results/050526/best_model_tag_performance.csv",index_col=0)

In [8]:
roberta_df.sort_values(by="SF1.1",ascending=False
            )

,Tag,SP,SR,SF1,RP,RR,RF1,SP.1,SR.1,SF1.1
11,Smoke,0.888,0.916,0.899,0.956,0.982,0.965,0.971,0.982,0.975
13,SubstanceUse,0.718,0.698,0.705,0.911,0.907,0.906,0.959,0.914,0.935
10,Recommendation,0.365,0.342,0.351,0.876,0.815,0.838,0.941,0.824,0.874
4,Financial,0.537,0.577,0.551,0.731,0.795,0.755,0.804,0.832,0.814
14,Transportation,0.553,0.616,0.577,0.705,0.783,0.735,0.788,0.836,0.804
6,Insurance,0.610,0.602,0.593,0.809,0.749,0.761,0.858,0.782,0.804
3,Employment,0.570,0.594,0.578,0.739,0.755,0.742,0.805,0.809,0.802
9,MentalHealth,0.545,0.520,0.531,0.742,0.810,0.771,0.761,0.797,0.774
8,Living,0.637,0.720,0.671,0.656,0.745,0.693,0.736,0.820,0.772
2,Education,0.585,0.606,0.579,0.706,0.726,0.697,0.756,0.757,0.741
